## Byte - Pair Encoding (BPE) Tokenizer  From Scratch

* Note this is the not so simple version and the notes are the same as those of the simple version: So I decided to just add the the code only here.

## A SImple BPE implementation
* Below is an implementation of this algorithm described above as a Python class that mimics the tiktoken Python user interface
*  Note that the encoding part above describes the original training step via `train()`; however, the `encode()` method works similarly (although it looks a bit more complicated because of the special token handling):

1) Split the input text into individual bytes
2) Repeatedly find & replace (merge) adjacent tokens (pairs) when they match any pair in the learned BPE merges (from highest to lowest "rank," i.e., in the order they were learned)
3) Continue merging untill no more merges can be applied
4) The final list of token IDs in the encoded output



In [2]:
# from collections import Counter, deque
# from functools import lru_cache
# import re
# import json

# class BPETokenizerSimple:
#   def __init__(self):
#     # Maps token_id to token_str (e.g: {11246: "some"})
#     self.vocab = {}
#     # Maps token_str to token_id (eg: {"some": 11246})
#     self.inverse_vocab = {}
#     # Dictionary of BPE merges {(token_id1, tokend_id2): merged_token_id}
#     self.bpe_merges = {}

#     # For the official openAI GPT-2 merges , use a rank dict
#     # of form {(string_A, string_B): rank}, where lower rank = higher priority
#     self.bpe_ranks = {}

#   def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
#     """
#     Train the BPE tokenizer from scratch.

#     Args:
#         text (str): The training text.
#         vocab_size (int): The desired vocabulary size.
#         allowed_special: (set): A set of special tokens to include.
#     """
#     # Preprocess: Replace spaces with    "Ġ"
#     # Note that "Ġ" is a particularity of the GPT-2 BPE implemetation
#     # EG: "Hello world" might be tokenized as ["Hello", "Ġworld"]
#     # (GPT-4 BPE would tokenize it as ["Hello", "world"])
#     processed_text = []
#     for i, char in enumerate(text):
#         if char == " " and i != 0:
#             processed_text.append("Ġ")
#         if char != " ":
#             processed_text.append(char)
#     processed_text = "".join(processed_text)
#     #  Initialize the vocab with  unique characters, includig "Ġ" if present
#     # Start with the first 256 ASCII caracters.
#     unique_chars = [chr (i) for i in range(256)]
#     unique_chars.extend(
#         char for char in sorted(set(processed_text))
#         if char not in unique_chars
#     )
#     if "Ġ" not in unique_chars:
#       unique_chars.append("Ġ")

#     self.vocab = {i: char for i, char in enumerate(unique_chars)}
#     self.inverse_vocab = {char: i for i, char in self.vocab.items()}

#     # Add allowed special tokens.
#     if allowed_special:
#       for token in allowed_special:
#         if token not in self.inverse_vocab:
#           new_id = len(self.vocab)
#           self.vocab[new_id] = token
#           self.inverse_vocab[token] = new_id


#     # tokenize the preocessed_text into token IDs
#     token_ids = [self.inverse_vocab[char] for char in processed_text]

#     # BPE steps 1-3 : Repetedly find and replace frequent pairs
#     for new_id in range(len(self.vocab), vocab_size):
#       pair_id = self.find_freq_pair(token_ids, mode="most")
#       if pair_id is None:
#         break
#       token_ids = self.replace_pair(token_ids, pair_id, new_id)
#       self.bpe_merges[pair_id] = new_id

#     # Build the vocabulary with merged tokens
#     for (p0, p1), new_id in self.bpe_merges.items():
#       merged_token = self.vocab[p0] + self.vocab[p1]
#       self.vocab[new_id] = merged_token
#       self.inverse_vocab[merged_token] = new_id

#   def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
#     """ Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

#         Args:
#             vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
#             bpe_merges_path (str): Path to the bpe_merges file (GPT-2 calls it 'vocab.bpe')
#     """
#     # Load vocabulary
#     with open(vocab_path, "r", encoding="utf-8") as file:
#       loaded_vocab = json.load(file)
#       # encoder.json is {token_str: id}; we want the id->str and str->id
#       self.vocab = {int(v): k for k , v in loaded_vocab.items()}
#       self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}

#       # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
#       if 'Ċ' not in self.inverse_vocab or self.inverse_vocab['Ċ'] != 198:
#         raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
#       # Must have <|endoftext|> at 50256
#       if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
#         raise KeyError("Vocabulary missing <|endoftext|> at id 50256")

#       # Provide a convenience alias for '\n' -> 198b
#       # keep the printable character 'Ċ' in vocab so BPE merges keep working

#       if "\n" not in self.inverse_vocab:
#         self.inverse_vocab["\n"] = self.inverse_vocab['Ċ']

#       if "\r" not in self.inverse_vocab:
#         if 201 in self.vocab:
#           self.inverse_vocab["\r"] = 201

#         else:
#           raise KeyError("Vocabulary missing carriage return token at id 201 ")

#       # Load GPT-2 merges and store ranks
#       self.bpe_ranks = {}
#       with open(bpe_merges_path, "r", encoding="utf-8") as file:
#         lines = file.readlines()
#         if lines and lines[0].startswith("#"):
#           lines = lines[1:]

#         rank = 0
#         for line in lines:
#           token1, *rest = line.strip().split()
#           if len(rest) != 1:
#             continue
#           token2 = rest[0]
#           if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
#             self.bpe_ranks[(token1, token2)] = rank
#             rank += 1
#           else:
#             # Safe to skip pairs whose symbols are not in vocab            #
#             pass

#   def encode(self, text, allowed_special=None):
#     """
#     Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.

#     Args:
#         text (str): the input text to encode
#         allowed_special (set or None):  Special tokens to allow passthrough. If None, special handling is disabled.
#     Returns:
#       List of token IDs
#     """
#     # --- This section is to mimic the tiltoken in terms of allowed special tokens -----
#     specials_in_vocab = [
#         tok for tok in self.inverse_vocab
#         if tok.startswith("<|") and tok.endswith("|>")
#     ]
#     if allowed_special is None:
#       # Nothing is allowed
#       disallowed = [tok for tok in specials_in_vocab if tok in text]
#       if disallowed:
#         raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

#       else:
#         # Some tokens are allowed (eg: we use this for <|endoftext|>)
#         disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
#         if disallowed:
#           raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
#     # -------------------------------------------------------------------

#     token_ids = []
#     # If some specials are allowed , split around them and passthrough those ids
#     if allowed_special is not None and len(allowed_special) > 0:
#       special_pattern = "(" + "|".join(
#           re.escape(tok) for tok in sorted(allowed_special, key=None, reverse=True)
#         ) + ")"
#       last_index = 0
#       for match in re.finditer(special_pattern, text):
#           prefix = text[last_index:match.start()]
#           token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally

#           special_token = match.group(0)
#           if special_token in self.inverse_vocab:
#             token_ids.append(self.inverse_vocab[special_token])
#           else:
#             raise ValueError(f"Special token {special_token} not found in vocabulary.")
#           last_index = match.end()

#       text = text[last_index:]  # remainder to process normally

#       # Extra guard for any other special literals left over
#       disallowed = [
#           tok for tok in self.inverse_vocab
#           if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special

#       ]
#       if disallowed:
#         raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

#   #----------------------- Newline and carriage return handling ------------------

#     tokens = []
#     parts = re.split(r'(\r\n|\r|\n)', text)
#     for part in parts:
#       if part  == "":
#         continue
#       if part == "\r\n":
#         tokens.append("\r")
#         tokens.append("\n")
#         continue
#       if part == "\r":
#         tokens.append("\r")
#         continue
#       if part == "\n":
#         tokens.append("\n")
#         continue


#       # Normal chunk without line break:
#       # If spaces precede a word, prefix the first with "Ġ" and add standalone trail the chunk(eg: before a newline ) add
#       # standalone "Ġ" tokens (tiktoken produces id 220 for "Ġ")
#       pending_spaces = 0
#       for m in re.finditer(r'( +)|(\S+)', part):
#         # r'( +)|(\S+)': This pattern matches one of two things, saving the result in a different group:
#         # Group 1 (( +)): Matches one or more spaces (e.g., ' ', ' ', ' '). This represents sequences of whitespace.
#         # Group 2 ((\S+)): Matches one or more non-space characters (e.g., 'Hello', 'world.'). This represents words or punctuation clumps.
#         if m.group(1) is not None:   #Match is one or more spaces
#           pending_spaces += len(m.group(1))
#         else:
#           word = m.group(2)   #(Match is a word, m.group(2) is not None)
#           if pending_spaces > 0:
#             tokens.append("Ġ" + word) # one leading space
#             for _ in range(pending_spaces - 1):
#               tokens.append("Ġ")
#             pending_spaces = 0
#           else:
#             tokens.append(word)
#       # Trailing spaces (no following word): ass standalone 'Ġ' tokens
#       for _ in range(pending_spaces):
#         tokens.append("Ġ")

#  #-----------------------------------------------------------------------------

#   # map tokens -> ids (BPE if needed)
#     for tok in tokens:
#       if tok in self.inverse_vocab:
#         token_ids.append(self.inverse_vocab[tok])
#       else:
#         token_ids.extend(self.tokenize_with_bpe(tok))
#     return token_ids

#   def tokenize_with_bpe(self, token):
#     """
#     Tokenize a single token using BPE merges.

#     Args:
#         token (str): The token to tokenize
#     Returns:
#         List [int]: The list of token IDs after applying BPE
#     """
#     # Tokenize the token into individual characters
#     token_ids = [self.inverse_vocab.get(char, None) for char in token]
#     if None in token_ids:
#       missing_chars = [char for char , tid in zip(token, token_ids) if tid is None]
#       raise ValueError(f"Characters not found in vocab: {missing_chars}")

#     # If we haven't loaded OpenAI's GPT2 merges, use the other approach we done for ourself.
#     if not self.bpe_ranks:
#       can_merge = True
#       while can_merge and len(token_ids) > 1:
#         can_merge = False
#         new_tokens = []
#         i = 0
#         while i < len(token_ids) - 1:
#           pair = (token_ids[i], token_ids[i+1])
#           if pair in self.bpe_merges:
#             merged_token_id = self.bpe_merges[pair]
#             new_tokens.append(merged_token_id)
#             print(f"Merged pair {pair} -> merged_token_id ('{self.vocab[merged_token_id]}')")
#             i += 2  # skip the next token as it's merged
#             can_merge = True
#           else:
#             new_tokens.append(token_ids[i])
#         if i < len(token_ids):
#           new_tokens.append(token_ids[i])
#         token_ids = new_tokens
#       return token_ids

#       # Otherwise do GPT-2 style merging with the ranks
#       # 1) Convert token_ids back to string "symbols" for each ID
#       symbols = [self.vocab[id_num] for id_num in token_ids]

#     # Repeatedly merge all occurrences of the lowse-rank pair
#     while True:
#       # Collect all adjacent pairs
#       pairs = set(zip(symbols, symbols[1:]))
#       if not pairs:
#         break

#       # Find the pair with the best (lowest) rank
#       min_rank = float("inf")
#       bigram = None
#       for p in pairs:
#         r = self.bpe_ranks.get(p, float("inf"))
#         if r < min_rank:
#           min_rank = r
#           bigram = p

#       # If no valid ranked pair is present, we're done
#       if bigram is None or bigram not in self.bpe_ranks:
#         break

#       # Merge all occurences of that pair
#       first, second = bigram
#       new_symbols = []
#       i =  0
#       while i < len(symbols):
#         # If we see (first, second) at position i, merge them
#         if i < len(symbols) -1 and symbols[i] == first and symbols[i+1] == second:
#           new_symbols.append(first + second)   # merge the symbol
#           i += 2
#         else:
#           new_symbols.append(symbols[i])
#           i += 1
#       symbols = new_symbols
#       if len(symbols) == 1:
#         break

#     # Finally , convert merged symnbols back to IDs
#     merged_ids = [self.inverse_vocab[sym] for sym in symbols]
#     return merged_ids

#   def decode(self, token_ids):
#     """
#     Decode a list of token IDs back into a sting

#     Args:
#         token_ids (List[int]): The list of token IDs to decode
#     Return:
#         str: The decoded string.
#     """
#     out = []
#     for tid in token_ids:
#       if tid not in self.vocab:
#         raise ValueError(f"Token ID {tid} not found in vocab")
#       tok = self.vocab[tid]

#       # Map GPT-2 special chars back to real chars
#       if tid == 198 or tok == "\n":
#         out.append("\n")
#       elif tid == 201 or tok == "\r":
#         out.append("\r")
#       elif tok.startswith("Ġ"):
#         out.append(" " + tok[1:])
#       else:
#         out.append(tok)
#     return "".join(out)


#   def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
#     """
#     Save the vocabulary and BPE merges to JSON files.

#     Args:
#         vocab_path (str): Path to save the vocabulary
#         bpe_merges_path (str): Path to save te BPE merges
#     """
#     # Save the vocabualry
#     with open(vocab_path, "w", encoding="utf-8") as file:
#       json.dump(self.vocab, file, ensure_ascii=False, indent=2)

#     # Save the BPE merges as a list of dictionaries
#     with open(bpe_merges_path, "w", encoding="utf-8") as file:

#       merges_list = [{"pair": list(pair), "new_id": new_id}
#                     for pair, new_id in self.bpe_merges.items()]
#       json.dump(merges_list, file, ensure_ascii=False, indent=2)

#   def load_vocab_and_merges(self, vocab, vocab_path, bpe_merges_path):
#     """
#     Load the vocabulary and BPE merges from JSON files.

#     Args:
#         vocab_path (str): Path to the vocabulary file
#         bpe_merge_path (str): Path to the BPE merges file.
#     """
#     # Load the vocabulary
#     with open(vocab_path, "r", encoding="utf-8") as file:
#       loaded_vocab = json.load(file)
#       self.vocab = {int(k): v for k , v in loaded_vocab.items()}
#       self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

#     # Load the BPE merges
#     with open(bpe_merges_path, "r", encoding="utf-8") as file:
#       merges_list = json.load(file)
#       for merge in merges_list:
#         pair = tuple(merge["pair"])
#         new_id = merge["new_id"]
#         self.bpe_merges[pair] = new_id

#   @lru_cache(maxsize=None)
#   def get_special_token_id(self, token):
#     return self.inverse_vocab.get(token, None)


#   @staticmethod
#   def find_freq_pair(token_ids, mode="most"):
#     pairs = Counter(zip(token_ids, token_ids[1:]))

#     if not pairs:
#       return None

#     if mode == "most":
#       return max(pairs.items(), key=lambda x: x[1])[0]
#     elif mode == "least":
#       return min(pairs.items(), key=lambda x: x[1])[0]
#     else:
#       raise ValueError("Invalid mode. CHoose 'most' or 'least'.")

#   @staticmethod
#   def replace_pair(token_ids, pair_id, new_id):
#     dq = deque(token_ids)
#     replaced = []

#     while dq:
#       current = dq.popleft()
#       if dq and (current , dq[0]) == pair_id:
#         replaced.append(new_id)

#         # Remove the 2nd token of the pair, 1st was already removed.
#         dq.popleft()
#       else:
#         replaced.append(current)
#     return replaced








## Training Encoding and Decoding

In [1]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # For the official OpenAI GPT-2 merges, use a rank dict:
        #  of form {(string_A, string_B): rank}, where lower rank = higher priority
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with "Ġ"
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including "Ġ" if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted(set(processed_text))
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:
                break
            token_ids = self.replace_pair(token_ids, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json is {token_str: id}; we want id->str and str->id
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}

        # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")

        # Must have <|endoftext|> at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")

        # Provide a convenience alias for '\n' -> 198
        # Keep printable character 'Ċ' in vocab so BPE merges keep working
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Load GPT-2 merges and store ranks
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]

            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Safe to skip pairs whose symbols are not in vocab
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.

        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.

        Returns:
            List of token IDs.
        """

        # ---- This section is to mimic tiktoken in terms of allowed special tokens ----
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            # Nothing is allowed
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            # Some spefic tokens are allowed (e.g., we use this for <|endoftext|>)
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        # -----------------------------------------------------------------------------

        token_ids = []
        # If some specials are allowed, split around them and passthrough those ids
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"

            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally

                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()

            text = text[last_index:]  # remainder to process normally

            # Extra guard for any other special literals left over
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")


        # ---- Newline and carriage return handling ----
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        tokens.append("Ġ" + word) # one leading space
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Trailing spaces (no following word): add standalone 'Ġ' tokens
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        # ---------------------------------------------------------------

        # Map tokens -> ids (BPE if needed)
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # If we haven't loaded OpenAI's GPT-2 merges, use my approach
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # Uncomment for educational purposes:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # Otherwise, do GPT-2-style merging with the ranks:
        # 1) Convert token_ids back to string "symbols" for each ID
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # Repeatedly merge all occurrences of the lowest-rank pair
        while True:
            # Collect all adjacent pairs
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # Find the pair with the best (lowest) rank
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # If no valid ranked pair is present, we're done
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # Merge all occurrences of that pair
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # If we see (first, second) at position i, merge them
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # Finally, convert merged symbols back to IDs
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # Map GPT-2 special chars back to real chars
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Save vocabulary
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # Save BPE merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # Load BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        pairs = Counter(zip(token_ids, token_ids[1:]))

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced

In [2]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
  for directory in search_dirs:
    file_path = os.path.join(directory, filename)
    if os.path.exists(file_path):
      print(f"File '{filename}' already exists in directory '{directory}'.")
      return file_path
  target_path = os.path.join(search_dirs[0], filename)
  try:
    response = requests.get(url, stram=True,  timeout=60)
    response.raise_for_status()
    with open(target_path, "wb") as out_file:
      for chunk in response.iter_content(chunk_size=8192):
        if chunk:
          out_file.write(chunk)
    print(f"Downloaded {filename} to {target_path}")
  except Exception as e:
    print(f"Failed to download {filename}. Error {e}")
  return target_path

verdict_path = download_file_if_absent(
  url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
  filename= "the-verdict.txt",
  search_dirs = ["./"]
)


with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()



File 'the-verdict.txt' already exists in directory './'.


* Next, let's initialize and train the BPE tokenizer with a vocabulary size of 1,000
* Note that the vocabulary size is already 256 by default due to the byte values discussed earlier, so we are only "learning" 744 vocabulary entries (if we consider the <|endoftext|> special token and the Ġ whitespace token; so, that's 742 to be precise)

* For comparison, the GPT-2 vocabulary is 50,257 tokens, the GPT-4 vocabulary is 100,256 tokens (cl100k_base in tiktoken), and GPT-4o uses 199,997 tokens (o200k_base in tiktoken); they have all much bigger training sets compared to our simple example text above


In [3]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

In [4]:
print(tokenizer.vocab)
print(len(tokenizer.vocab))


{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '\x1e', 31: '\x1f', 32: ' ', 33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93: ']', 94: '^', 95: '_', 96: '`', 97: 'a', 98: 'b', 99: 'c', 100: 'd', 101: 'e'

 This vocabulary is created by merging 742 times `(= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742)`



In [5]:
print(len(tokenizer.bpe_merges))

742


*  this means the first 256 entries are single-character tokens

* Next, let's use the created merges via the `encode` method to encode some text:


In [6]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


*    Iterating over each token ID can give us a better understanding of how the token IDs are decoded via the vocabulary:



In [7]:
print("Number of characters:", len(input_text))
print("Number of token IDs", len(token_ids))

Number of characters: 42
Number of token IDs 20


*     As we can see, most token IDs represent 2-character subwords; that's because the training data text is very short with not that many repetitive words, and because we used a relatively small vocabulary size
*     As a summary, calling `decode(encode())` should be able to reproduce arbitrary input texts:




In [8]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

In [9]:
tokenizer.decode(tokenizer.encode("This is some text with  \n newline characters."))

'This is some text with  \n newline characters.'

### Saving and loading the Tokenizer


In [10]:
# Save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [11]:
# Load the tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

*    The loaded tokenizer should be able to produce the same results as before:



In [ ]:
print(tokenizer2.decode(token_ids))

In [12]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters")
)

'This is some text with \n newline characters'

## Loading the original GPT-2 BPE tokenizer from OpenAI

In [13]:
# Download files if not already present in this directory

# Define the directories to search and the files to download
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Ensure directories exist and download files if needed
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)


Failed to download vocab.bpe. Error Session.request() got an unexpected keyword argument 'stram'
Failed to download encoder.json. Error Session.request() got an unexpected keyword argument 'stram'


In [14]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"],
    bpe_merges_path=paths["vocab.bpe"]
)

FileNotFoundError: [Errno 2] No such file or directory: 'ch02/02_bonus_bytepair-encoder/gpt2_model/encoder.json'


    The vocabulary size should be 50257 as we can confirm via the code below:



In [ ]:
len(tokenizer_gpt2.vocab)

* We can now use the GPT-2 tokenizer via our `BPETokenizerSimple` object:

In [ ]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

In [ ]:
print(tokenizer_gpt2.decode(token_ids))

## Conclusion

* That's it! That's how BPE works in a nutshell, complete with a training method for creating new tokenizers or loading the GPT-2 tokenizer vocabular and merges from the original OpenAI GPT-2 model